# ML-based downscaling of future climate scenarios

## 1. Setting up

During this workshop we are going to train a deep learning climate downscaling model based on a UNet architecture. We are going to use `EC-Earth3-Veg` global climate model data, interpolated to 1° resolution, as the low resolution input data (**predictors**), and `HCLIM43-ALADIN` regional climate model data, interpolated to 0.1° resolution, as the high resolution output data (**target**). The regional model is forced by `EC-Earth3-Veg` here.

A graphical processing unit (GPU) is necessary to train such a model within reasonable time. Start the exercises by connecting to a T4 GPU runtime. Click on a dropdown menu next to `Connect` button in the upper right corner, select `Change runtime type`, then `T4 GPU` and press `Save`. After selecting the T4 GPU runtime, proceed to connecting by pressing `Connect T4` button.

You will be given an access to a virtual machine with a T4 GPU and a small amount of temporary storage. Verify that the connection was successful by executing the next block:

In [ ]:
! nvidia-smi

If the connection has been successful, you should be able to see the output of `nvidia-smi` diagnostics utility showing the `Tesla T4` GPU is available.

## 2. Data download

Download the training data:

In [ ]:
! rsync -av rsync://exporter.nsc.liu.se/cdd718e2908646239133f4e7d8455cd0 .

Open the `Files` side panel to access the downloaded files.

You should be able to see `predictors` and `target` folders with multiple `.nc` data files.

What you have downloaded is SSP3.7-0 data for the 2015-2100 period from both EC-Earth3-Veg and HCLIM models, both centered around Denmark.

Let's install a few extra packages and try to inspect a few timesteps from the downloaded climate data:

## 3. Extra package installation

In [ ]:
! pip install netCDF4 scitools-iris palettable

## 4. Climate data plotting

Import a few Python libraries to help with the plotting:

In [ ]:
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import palettable
import cartopy.feature as cfeature

mpl.rcParams['font.size'] = 10
mpl.rcParams['font.family'] = 'sans-serif'
cm = 1/2.54

Plot surface temperature `tas` variable from EC-Earth3-Veg model on a single day:

In [ ]:
# Open the dataset
ds = xr.open_dataset('predictors/tas_EC-Earth3-Veg_2015-2100.nc')

In [ ]:
# Select first timestep and convert to Celsius
tas_c = ds["tas"].isel(time=0) - 273.15

In [ ]:
# Create map
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

# Plot
tas_c.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=palettable.colorbrewer.diverging.RdYlBu_10_r.mpl_colormap,
    vmin=-20,
    vmax=10,
    cbar_kwargs={"label": "Surface temperature (°C)"}
)

# Add coastlines
ax.coastlines(resolution='10m')

# Set the title
ax.set_title("")

# Show the plot
plt.show()

Now let's compare with the high resolution data from the regional climate model HCLIM:

In [ ]:
# Open the dataset
ds = xr.open_dataset('target/tas_remapped_HCLIM_2015-2100.nc')

In [ ]:
# Select first timestep and convert to Celsius
tas_c = ds["tas"].isel(time=0) - 273.15

In [ ]:
# Create map
fig, ax = plt.subplots(subplot_kw={"projection": ccrs.PlateCarree()})

# Plot
tas_c.plot(
    ax=ax,
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=palettable.colorbrewer.diverging.RdYlBu_10_r.mpl_colormap,
    vmin=-20,
    vmax=10,
    cbar_kwargs={"label": "Surface temperature (°C)"}
)

# Add coastlines
ax.coastlines(resolution='10m')

# Set the title
ax.set_title("")

# Show the plot
plt.show()

Essentially, the goal of downscaling is to start with low resolution climate data, and increase the resolution to a desired level. While regional climate models run an additional climate model simulation on a finer grid (which requires a lot of computational resources), we are going to train a neural network that will learn a relation between the low and high resolution data, so that it can be used later to produce new high resolution data.

## 5. Standardization of the training data

Now let's download the training scripts and start preparing the files:

In [ ]:
! rsync -av rsync://exporter.nsc.liu.se/f944fc0abc944c83a653d8dfb3b19ee9 .

In [ ]:
! ls

We are going to prepare some data folders:

In [ ]:
! mkdir data

In [ ]:
! mv predictors ./data/
! mv target ./data/

In [ ]:
! mkdir -p data/predictors/standardized
! mkdir -p data/target/standardized

In [ ]:
! mkdir output

Before starting the training, we need to standardize the data first to make the training faster and more stable. To do that, we are going to compute monthly means and monthly standard deviations for every variable and every grid point. To standardize the data, we subtract the corresponding mean value and divide by the corresponding standard deviation:

$$
x' = \frac{x - \mu}{\sigma}
$$

where $x'$ is a standardized value $x$, $\mu$ is the monthly mean, and $\sigma$ is the monthly standard deviation.

Thus, the neural network learns to produce standardized high resolution data. To produce the high resolution data in the original space (or units - `K` for surface temperature), we are going to perform the inverse of standardization:

$$
x = \sigma x' + \mu
$$

To perform the standardization, you can use the methods implemented in our `detex` code:

In [ ]:
import os
from detex.standardization import get_local_monthly_stats, standardize_local_monthly, scale_static_field

Standardize the predictors using statistics from 2015-2065 period (training period).

In [ ]:
GCM_name = 'EC-Earth3-Veg'
GCM_period_start = '2015'
GCM_period_end = '2100'
start_date = '2015-01-01'
end_date = '2064-12-31'
rel_path = 'data/predictors'
std_path = 'data/predictors/standardized'
vars = ['ta', 'ua', 'va', 'hus', 'zg']
levels = [1000, 850, 700, 500]

for var_name in vars:
    for level in levels:
        ds_name = f"{var_name}_{level}_{GCM_name}_{GCM_period_start}-{GCM_period_end}.nc"
        get_local_monthly_stats(
            ds_name=ds_name, rel_path=rel_path, std_path=std_path, var_name=var_name, level=level,
            start_date=start_date, end_date=end_date, dataset_name=GCM_name
        )
        stats_file = os.path.join(std_path, f'local_monthly_mean_std_{var_name}_{level}_{GCM_name}_{start_date}_{end_date}.npy')
        standardize_local_monthly(
            ds_name=ds_name, rel_path=rel_path, std_path=std_path, stats_file=stats_file,
            var_name=var_name, x_name='lat', y_name='lon'
        )

Scale the orography to [0, 1] range:

In [ ]:
scale_static_field(
    ds_name='data/predictors/orog_remapped_HCLIM.nc',
    out_name='data/predictors/scaled.orog_remapped_HCLIM.nc',
    var_name='orog'
)

Move the static data (scaled orography and the land-sea mask to the folder with standardized predictors):

In [ ]:
! cp data/predictors/scaled.orog_remapped_HCLIM.nc data/predictors/standardized/
! cp data/predictors/land_sea_mask_remapped_HCLIM.nc data/predictors/standardized/

Standardize the target data:

In [ ]:
RCM_name = 'remapped_HCLIM'
RCM_period_start = '2015'
RCM_period_end = '2100'
start_date = '2015-01-01'
end_date = '2064-12-31'
rel_path = 'data/target'
std_path = 'data/target/standardized/'
var = 'tas'

ds_name = f"{var}_{RCM_name}_{RCM_period_start}-{RCM_period_end}.nc"
get_local_monthly_stats(
    ds_name=ds_name, rel_path=rel_path, std_path=std_path, var_name=var,
    start_date=start_date, end_date=end_date, dataset_name=RCM_name
)
stats_file = os.path.join(std_path, f'local_monthly_mean_std_{var}_{RCM_name}_{start_date}_{end_date}.npy')
standardize_local_monthly(
    ds_name=ds_name, rel_path=rel_path, std_path=std_path, stats_file=stats_file,
    var_name=var, x_name='lat', y_name='lon'
)

## 6. Training

Now we are going to start the training. The period of 2015-2065 is used for the training and 2065-2070 for validation. We will use 2070-2100 for later testing.



In [ ]:
! python training.py \
    -c training-config.toml \
    -d cuda \
    -n 2

As you can see, the training can take a while. Instead of waiting, you can stop the training by clicking the "stop" button on the left of the cell, and download a trained model checkpoint. Running the predictions with a trained model takes much less time.

Download the checkpoint:

In [ ]:
! rsync -av rsync://exporter.nsc.liu.se/a729e33e74134346969793b83e19336b ./output/

# 7. Inference

Now we are going to run the predictions for the 2070-2100 period:

In [ ]:
! python inference.py \
    -c inference-config.toml \
    -d cuda \
    -n 2

We can finally analyze the ML-downscaling predictions and compare them to high resolution data from 2070-2100 period. You are free to come up with your own ways to evaluate the performance of the ML-downscaling method, but below you can find a few examples.

## 8. Data analysis

Let's install `cdo` (Climate Data Operators) tool to quickly manipulate our climate data:

In [ ]:
! apt-get update
! apt-get install -y cdo

We can select just the test period (2070-2100) for easier comparison with the ML predictions:

In [ ]:
! cdo selyear,2070/2100 \
        data/target/tas_remapped_HCLIM_2015-2100.nc \
        data/target/tas_remapped_HCLIM_2070-2100.nc

In [ ]:
! cdo -L -sellonlatbox,7,14,53,60 -selyear,2070/2100 \
        data/predictors/tas_EC-Earth3-Veg_2015-2100.nc \
        data/predictors/tas_EC-Earth3-Veg_2070-2100.nc

### Downscaling example

Let's plot a single timestep to see how the ML-downscaling compares to the reference data:

In [ ]:
ds_ml = xr.open_dataset('output/Predictions_tas_EC-Earth3-Veg_2070-2100.nc')
ds_hclim = xr.open_dataset('data/target/tas_remapped_HCLIM_2070-2100.nc')
ds_ece = xr.open_dataset('data/predictors/tas_EC-Earth3-Veg_2070-2100.nc')

In [ ]:
# List of datasets and titles
datasets = [ds_hclim, ds_ml, ds_ece]
titles = ["HCLIM (0.1°)", "ML-downscaling (0.1°)", "EC-Earth3-Veg (1°)"]

# Create subplots (1 row, 3 columns)
fig, axes = plt.subplots(
    1, 3,
    figsize=(7, 5),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

meshes = []

for ax, ds, title in zip(axes, datasets, titles):
    # Convert temperature to Celsius
    tas_c = ds["tas"].isel(time=15) - 273.15

    mesh = tas_c.plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        x="lon",
        y="lat",
        cmap=ListedColormap(palettable.colorbrewer.diverging.RdYlBu_10_r.mpl_colors),
        vmin=-15,
        vmax=15,
        add_colorbar=False
    )

    ax.coastlines()
    ax.set_title(title)
    meshes.append(mesh)

# Single horizontal colorbar
cbar = fig.colorbar(
    meshes[0],
    ax=axes,
    orientation="horizontal",
    pad=0.05,
    aspect=45,
    ticks=np.arange(-15, 21, 6),
    extend='both',
    shrink=0.8
)
cbar.set_label("Temperature (°C)")

plt.savefig('output/downscaling-example.png', dpi=300, bbox_inches='tight')
plt.show()

### Mean annual temperature change

Now we are going to check whether the ML-downscaling model reproduces the warming trend during the last 30 years.

In [ ]:
ds = xr.open_dataset("data/predictors/tas_EC-Earth3-Veg_2015-2100.nc")

ds_sel = ds.sel(lon=slice(7, 14), lat=slice(60, 53))
tas = ds_sel["tas"].mean(dim=["lat", "lon"])
tas = tas.resample(time="YS").mean()
tas.to_dataframe(name="tas").reset_index().to_csv(
    "data/predictors/yearly_mean_tas_EC-Earth3-Veg_2015-2100.csv",
    index=False
)

In [ ]:
ds = xr.open_dataset("data/target/tas_remapped_HCLIM_2015-2100.nc")

tas = ds["tas"].mean(dim=["lat", "lon"])
tas = tas.resample(time="YS").mean()
tas.to_dataframe(name="value").reset_index().to_csv(
    "data/target/yearly_mean_tas_remapped_HCLIM_2015-2100.csv",
    index=False
)

In [ ]:
ds = xr.open_dataset("output/Predictions_tas_EC-Earth3-Veg_2070-2100.nc")

tas = ds["tas"].mean(dim=["lat", "lon"])
tas = tas.resample(time="YS").mean()
tas.to_dataframe(name="value").reset_index().to_csv(
    "output/yearly_mean_Predictions_tas_EC-Earth3-Veg_2070-2100.csv",
    index=False
)

In [ ]:
df_hclim = pd.read_csv(
    "data/target/yearly_mean_tas_remapped_HCLIM_2015-2100.csv",
    parse_dates=["time"]
)

df_ece = pd.read_csv(
    "data/predictors/yearly_mean_tas_EC-Earth3-Veg_2015-2100.csv",
    parse_dates=["time"]
)

df_ml = pd.read_csv(
    "output/yearly_mean_Predictions_tas_EC-Earth3-Veg_2070-2100.csv",
    parse_dates=["time"]
)

# plot
plt.plot(df_hclim["time"], df_hclim["value"] - 273.15, label="HCLIM")
plt.plot(df_ece["time"], df_ece["tas"] - 273.15, label="EC-Earth3-Veg")
plt.plot(df_ml["time"], df_ml["value"] - 273.15, label="ML-downscaling")
plt.axvline(pd.Timestamp("2070-01-01"), color="black", linestyle="--")

plt.xlabel("Year")
plt.ylabel("Mean temperature, °C")
plt.legend()
plt.savefig('output/mean-annual-temperature.png', dpi=300, bbox_inches='tight')
plt.show()

### Seasonal root-mean squared error

Here we will examine seasonally averaged root-mean squared error (RMSE) of daily mean temperature (relative to the regional climate model):

In [ ]:
! cdo -L sqrt -yseasmean -pow,2 -sub \
        output/Predictions_tas_EC-Earth3-Veg_2070-2100.nc \
        data/target/tas_remapped_HCLIM_2070-2100.nc \
        output/seasonal_rmse_Predictions_tas_EC-Earth3-Veg_2070-2100.nc

In [ ]:
ds = xr.open_dataset('output/seasonal_rmse_Predictions_tas_EC-Earth3-Veg_2070-2100.nc')

In [ ]:
seasons = ["DJF", "MAM", "JJA", "SON"]

fig, axes = plt.subplots(
    2, 2,
    figsize=(6, 6),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

for i, ax in enumerate(axes.flat):

    ds["tas"].isel(time=i).plot(
        ax=ax,
        transform=ccrs.PlateCarree(),
        x="lon",
        y="lat",
        cmap=ListedColormap(palettable.cmocean.sequential.Thermal_6.mpl_colors),
        vmin=0,
        vmax=3,
        add_colorbar=False
    )

    ax.coastlines()
    ax.set_title(seasons[i])

# single colorbar for all plots
cbar = fig.colorbar(
    axes.flat[0].collections[0],
    ax=axes,
    orientation="horizontal",
    pad=0.05,
    extend='max'
)

cbar.set_label("RMSE temperature (°C)")

plt.savefig('output/temperature-rmse.png', dpi=300, bbox_inches='tight')
plt.show()

### Comparison of future climatologies

Finally, let's look at how the winter and summer climatologies of the 2070-2100 period compare to the reference data:

In [ ]:
! cdo -L -yseasmean -selyear,2070/2100 \
        data/target/tas_remapped_HCLIM_2015-2100.nc \
        data/target/yseasmean_tas_remapped_HCLIM_2070-2100.nc

In [ ]:
! cdo -L -yseasmean -sellonlatbox,7,14,53,60 -selyear,2070/2100 \
        data/predictors/tas_EC-Earth3-Veg_2015-2100.nc \
        data/predictors/yseasmean_tas_EC-Earth3-Veg_2070-2100.nc

In [ ]:
! cdo -L -yseasmean -selyear,2070/2100 \
        output/Predictions_tas_EC-Earth3-Veg_2070-2100.nc \
        output/yseasmean_Predictions_tas_EC-Earth3-Veg_2070-2100.nc

In [ ]:
ds_hclim = xr.open_dataset('data/target/yseasmean_tas_remapped_HCLIM_2070-2100.nc') - 273.15
ds_ece = xr.open_dataset('data/predictors/yseasmean_tas_EC-Earth3-Veg_2070-2100.nc') - 273.15
ds_ml = xr.open_dataset('output/yseasmean_Predictions_tas_EC-Earth3-Veg_2070-2100.nc') - 273.15

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(24*cm, 9*cm),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

meshes = []

vmin = -6
vmax = 9
step = 3
cmap = ListedColormap(palettable.colorbrewer.diverging.RdYlBu_10_r.mpl_colors)

mesh = ds_hclim["tas"].isel(time=0).plot(
    ax=axes[0],
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    add_colorbar=False
)

axes[0].coastlines()
axes[0].set_title(
    "HCLIM\n(Dec-Feb)"
)
meshes.append(mesh)

mesh = ds_ece["tas"].isel(time=0).plot(
    ax=axes[1],
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    add_colorbar=False
)

axes[1].coastlines()
axes[1].set_title(
    "EC-Earth3-Veg\n(Dec-Feb)"
)
meshes.append(mesh)

mesh = ds_ml["tas"].isel(time=0).plot(
    ax=axes[2],
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    add_colorbar=False
)

axes[2].coastlines()
axes[2].set_title(
    "ML downscaling\n(Dec-Feb)"
)
meshes.append(mesh)

# Single horizontal colorbar
cbar = fig.colorbar(
    meshes[0],
    ax=axes,
    orientation="horizontal",
    pad=0.05,
    aspect=45,
    ticks=np.arange(vmin, vmax + step, step),
    extend=None,
    shrink=0.6
)
cbar.set_label("Temperature (°C)")

plt.savefig('output/winter-climatology.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(
    1, 3,
    figsize=(24*cm, 9*cm),
    subplot_kw={"projection": ccrs.PlateCarree()}
)

meshes = []

vmin = 8
vmax = 23
step = 3
cmap = ListedColormap(palettable.colorbrewer.diverging.RdYlBu_10_r.mpl_colors)

mesh = ds_hclim["tas"].isel(time=2).plot(
    ax=axes[0],
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    add_colorbar=False
)

axes[0].coastlines()
axes[0].set_title(
    "HCLIM\n(Jun-Aug)"
)
meshes.append(mesh)

mesh = ds_ece["tas"].isel(time=2).plot(
    ax=axes[1],
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    add_colorbar=False
)

axes[1].coastlines()
axes[1].set_title(
    "EC-Earth3-Veg\n(Jun-Aug)"
)
meshes.append(mesh)

mesh = ds_ml["tas"].isel(time=2).plot(
    ax=axes[2],
    transform=ccrs.PlateCarree(),
    x="lon",
    y="lat",
    cmap=cmap,
    vmin=vmin,
    vmax=vmax,
    add_colorbar=False
)

axes[2].coastlines()
axes[2].set_title(
    "ML downscaling\n(Jun-Aug)"
)
meshes.append(mesh)

# Single horizontal colorbar
cbar = fig.colorbar(
    meshes[0],
    ax=axes,
    orientation="horizontal",
    pad=0.05,
    aspect=45,
    ticks=np.arange(vmin, vmax + step, step),
    extend=None,
    shrink=0.6
)
cbar.set_label("Temperature (°C)")

plt.savefig('output/summer-climatologies.png', dpi=300, bbox_inches='tight')
plt.show()

As an alternative to running the inference, you can download the predictions that I produced beforehand:

In [ ]:
! rsync -av rsync://exporter.nsc.liu.se/9a1fae5ccfdd4481898836d3ddec5a1b ./output/

After downloading the predictions, you can go back to section 8 (Data analysis) and run the analysis with the downloaded data.